# Momentum Factor Research

This notebook walks through a **complete factor research workflow** using Factorium:

1. Load data from Binance
2. Explore the AggBar data structure
3. Build momentum factors (code-based & expression-based)
4. Visualize factor behavior
5. Run IC (Information Coefficient) analysis
6. Analyze quantile returns
7. Run a vectorized backtest
8. Generate a quick report

**Prerequisites**: `pip install factorium` (or `uv add factorium`)

## 1. Setup & Data Loading

We'll download 30 days of 1-minute futures data for 10 crypto symbols from Binance Vision.

In [ ]:
from factorium import BinanceDataLoader, ResearchSession
from factorium.factors import FactorAnalyzer, CompositeFactor

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
SYMBOLS = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "AVAXUSDT", "DOTUSDT", "CAKEUSDT",
]

loader = BinanceDataLoader()

agg = loader.load_aggbar(
    symbols=SYMBOLS,
    data_type="aggTrades",
    market_type="futures",
    futures_type="um",
    days=30,
    bar_type="time",
    interval=60_000,       # 1-minute bars
)

print(f"Loaded {len(agg):,} bars")
print(f"Symbols: {agg.symbols}")
print(f"Columns: {agg.cols}")

## 2. Exploring the AggBar

`AggBar` is a **multi-symbol OHLCV container** stored in long format. You can inspect it, convert to Polars/Pandas, and slice by time or symbols.

In [ ]:
# Summary info per symbol
agg.info()

In [ ]:
# View raw data as Polars DataFrame
agg.to_polars().head(10)

In [ ]:
# Slice by symbols — creates a new AggBar
btc_eth = agg.slice(symbols=["BTCUSDT", "ETHUSDT"])
print(f"Sliced AggBar: {btc_eth.symbols}, {len(btc_eth):,} bars")

## 3. Building Momentum Factors

### 3.1 Code-Based Factor Construction

Extract a column from `AggBar` with `agg["close"]` to get a `Factor` object, then chain time-series and cross-sectional operators.

In [ ]:
close = agg["close"]
volume = agg["volume"]

# Simple momentum: percentage change over 60 periods
momentum_60 = close.ts_delta(60) / close.ts_shift(60)
momentum_60.name = "momentum_60"

# Short-term momentum (20 periods)
momentum_20 = close.ts_delta(20) / close.ts_shift(20)
momentum_20.name = "momentum_20"

# Volatility-adjusted momentum
volatility = (close.ts_delta(1) / close.ts_shift(1)).ts_std(60)
vol_adj_momentum = momentum_60 / volatility
vol_adj_momentum.name = "vol_adj_momentum"

# Cross-sectional rank (normalized to [0, 1])
momentum_rank = momentum_60.cs_rank()
momentum_rank.name = "momentum_rank"

print("Factor shapes:")
print(f"  momentum_60:      {len(momentum_60):,} rows")
print(f"  vol_adj_momentum: {len(vol_adj_momentum):,} rows")
print(f"  momentum_rank:    {len(momentum_rank):,} rows")

### 3.2 Expression-Based Factor Construction

You can also define factors using string expressions via `ResearchSession.create_factor()`. This is useful for rapid experimentation.

In [ ]:
session = ResearchSession(agg, default_frequency="1m")

# Same momentum factor, defined via expression
mom_expr = session.create_factor(
    "ts_delta(close, 60) / ts_shift(close, 60)",
    name="momentum_60_expr",
)

# A more complex expression: MA crossover signal
ma_cross = session.create_factor(
    "ts_mean(close, 10) - ts_mean(close, 30)",
    name="ma_crossover",
)

print(f"Expression-based momentum: {len(mom_expr):,} rows")
print(f"MA crossover signal:       {len(ma_cross):,} rows")

## 4. Visualizing Factors

Use the `.plot` accessor on Factor objects to produce time series, heatmaps, and distributions.

In [ ]:
# Time series of momentum for a few symbols
momentum_60.plot()
plt.show()

## 5. IC (Information Coefficient) Analysis

The **IC** measures the rank correlation between factor values and subsequent returns. A consistently positive (or negative) IC indicates predictive power.

### 5.1 Single-Period IC

In [ ]:
# Use ResearchSession for streamlined analysis
signal = momentum_rank  # Use ranked momentum as our signal

analysis = session.analyze(signal, periods=1)

print("IC Summary:")
print(analysis.ic_summary)
print()
print(f"IC Series shape: {analysis.ic_series.shape}")
print(analysis.ic_series.head())

### 5.2 Multi-Horizon IC Analysis

To understand how quickly a factor's signal decays, we compute IC across multiple forward horizons.

In [ ]:
# Compute IC for multiple horizons
horizons = [1, 5, 10, 20, 60]
ic_results = {}

for h in horizons:
    result = session.analyze(signal, periods=h)
    ic_results[h] = result.ic_summary.get(h, {})

# Display IC decay table
ic_decay_df = pd.DataFrame(ic_results).T
ic_decay_df.index.name = "horizon"
print("IC Decay Analysis:")
ic_decay_df

In [ ]:
# Plot IC decay curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ic_decay_df.index, ic_decay_df["mean_ic"], "o-", linewidth=2, markersize=8)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Forward Horizon (periods)")
ax.set_ylabel("Mean IC")
ax.set_title("IC Decay Curve — Momentum Factor")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Quantile Analysis

Split the cross-section into quantiles by factor value and examine the mean returns of each group. A monotonic pattern (Q1 < Q2 < ... < Q5) indicates strong linear predictive power.

In [ ]:
# Use FactorAnalyzer directly for more control
analyzer = FactorAnalyzer(signal, agg, quantiles=5)

# Prepare data: align factor values with forward returns
analyzer.prepare_data(price_col="close", periods=[1])

# Quantile returns
analyzer.plot_quantile_returns(quantiles=5, period=1)
plt.show()

In [ ]:
# Cumulative returns by quantile (includes Long-Short portfolio)
analyzer.plot_cumulative_returns(quantiles=5, period=1, long_short=True)
plt.show()

In [ ]:
# IC time series plot
analyzer.plot_ic(period=1, method="rank", plot_type="ts")
plt.show()

## 7. Backtesting

Run a **market-neutral vectorized backtest** using the momentum signal. The backtester:
- Converts signals to portfolio weights (cross-sectional normalization)
- Handles transaction costs
- Tracks equity, returns, and positions over time

In [ ]:
# Market-neutral backtest
result = session.backtest(
    signal,
    neutralization="market",
    transaction_cost=0.0003,
)

# Key performance metrics
print("Backtest Metrics:")
for key, val in result.metrics.items():
    if isinstance(val, float):
        print(f"  {key:25s}: {val:>10.4f}")
    else:
        print(f"  {key:25s}: {val}")

In [ ]:
# Plot equity curve
equity = result.equity_curve.to_pandas()
equity["timestamp"] = pd.to_datetime(equity["end_time"], unit="ms")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(equity["timestamp"], equity["total_value"], linewidth=1.5)
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Equity")
ax.set_title("Momentum Factor — Equity Curve (Market Neutral)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot returns distribution
returns = result.returns.to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(returns["return"].dropna(), bins=100, alpha=0.75, edgecolor="black", linewidth=0.5)
ax.axvline(x=0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Return")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Portfolio Returns")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Quick Report

`ResearchSession.quick_report()` combines IC analysis and backtesting into a single text summary — great for rapid iteration.

In [ ]:
report = session.quick_report(signal)
print(report)

## 9. Summary & Next Steps

In this notebook we:
- Loaded multi-symbol 1-minute data from Binance using `BinanceDataLoader`
- Built momentum factors via both **code-based** and **expression-based** methods
- Visualized factor behavior with time series, distributions, and heatmaps
- Computed IC and analyzed its decay across multiple horizons
- Ran a market-neutral backtest and examined performance metrics

**Next notebooks to explore:**
- `02_mean_reversion_factor.ipynb` — Mean reversion with volatility normalization
- `03_data_loading_and_exploration.ipynb` — Deep dive into AggBar and data loading
- `04_multi_factor_combination.ipynb` — Combine multiple factors and compare performance